# V3-0 — Freeze, audit and A2-MP reproduction

This notebook freezes the V2 reference, audits leakage and distributions, and retrains only the A2-MP trainable head from its frozen feature cache. It never overwrites V2 artifacts.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import random
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models

DATA_ROOT = Path(r'P:\NexarCollisionData')
MODEL_ROOT = DATA_ROOT / 'models_v2'
INFERENCE_ROOT = DATA_ROOT / 'inference_v2'
REPORT_ROOT = DATA_ROOT / 'reports_v3'
V3_MODEL_ROOT = DATA_ROOT / 'models_v3'
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
V3_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

VIDEO_MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
SPLIT_PATH = DATA_ROOT / 'metadata_split_v1.csv'
SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2_multipos.csv'
FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2_multipos.csv'
FEATURE_CACHE_PATH = DATA_ROOT / 'processed_v2' / 'resnet18_imagenet_features_v2_multipos_16x224x320.pt'
A2MP_CHECKPOINT_PATH = MODEL_ROOT / 'resnet18_meanmax_pooling_frozen_multipos_best.pt'
A2MP_CLIP_METRICS_PATH = MODEL_ROOT / 'resnet18_meanmax_pooling_frozen_multipos_clip_metrics.json'
A2MP_FULL_METRICS_PATH = INFERENCE_ROOT / 'a2_multipos_validation_sliding_metrics.json'
A2MP_FULL_PREDICTIONS_PATH = INFERENCE_ROOT / 'a2_multipos_validation_sliding_video_predictions.csv'
A2MP_CALIBRATION_PATH = INFERENCE_ROOT / 'a2_multipos_calibration_report.json'
A2MP_ERROR_PATH = INFERENCE_ROOT / 'a2_multipos_final_error_analysis.csv'

SEED = 42
NUM_FRAMES = 16
FEATURE_DIM = 512
HEAD_BATCH_SIZE = 64
EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
V2_MINIMUM_RECALL = 0.80
V3_MINIMUM_RECALL = 0.85
RUN_A2MP_HEAD_REPRODUCTION = True
RUN_FULL_MP4_REPRODUCTION = False
REPLAY_V2_INITIAL_FEATURE_EXTRACTION_RNG = True

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

def current_git_commit() -> str:
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return 'not_available'

required = [VIDEO_MANIFEST_PATH, SPLIT_PATH, SEQUENCE_MANIFEST_PATH, FRAME_INDEX_PATH, FEATURE_CACHE_PATH, A2MP_CHECKPOINT_PATH, A2MP_CLIP_METRICS_PATH, A2MP_FULL_METRICS_PATH]
for item in required:
    assert item.is_file(), 'Missing required V2 artifact: {}'.format(item)

print({'device': str(torch.device('cuda' if torch.cuda.is_available() else 'cpu')), 'python': sys.version.split()[0], 'torch': torch.__version__, 'data_root': str(DATA_ROOT)})


{'device': 'cpu', 'python': '3.13.2', 'torch': '2.13.0+cpu', 'data_root': 'P:\\NexarCollisionData'}


In [2]:
# 1) Immutable V2 reference inventory with SHA-256
def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(block_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

artifact_specs = [
    ('video_manifest', VIDEO_MANIFEST_PATH, '07_v2_data_audit_manifest.ipynb'),
    ('fixed_video_split', SPLIT_PATH, '01_video_data_split.ipynb'),
    ('multipos_sequence_manifest', SEQUENCE_MANIFEST_PATH, '18_v2_multipos_sequence_manifest.ipynb'),
    ('multipos_frame_cache_index', FRAME_INDEX_PATH, '19_v2_multipos_extract_sequence_frames.ipynb'),
    ('multipos_feature_cache', FEATURE_CACHE_PATH, '20_v2_multipos_resnet18_meanmax_pooling.ipynb'),
    ('a2mp_checkpoint', A2MP_CHECKPOINT_PATH, '20_v2_multipos_resnet18_meanmax_pooling.ipynb'),
    ('a2mp_clip_metrics', A2MP_CLIP_METRICS_PATH, '20_v2_multipos_resnet18_meanmax_pooling.ipynb'),
    ('a2mp_full_mp4_metrics', A2MP_FULL_METRICS_PATH, '21_v2_a2_multipos_sliding_window_evaluation.ipynb'),
    ('a2mp_full_mp4_predictions', A2MP_FULL_PREDICTIONS_PATH, '21_v2_a2_multipos_sliding_window_evaluation.ipynb'),
    ('a2mp_calibration', A2MP_CALIBRATION_PATH, '23_v2_a2mp_calibration_threshold.ipynb'),
    ('a2mp_error_analysis', A2MP_ERROR_PATH, '24_v2_a2mp_final_error_analysis_inference.ipynb')
]

artifact_rows = []
for name, path, source_notebook in artifact_specs:
    row = {'artifact_name': name, 'path': str(path), 'source_notebook': source_notebook, 'git_commit': current_git_commit(), 'freeze_policy': 'reference_only__do_not_overwrite'}
    if path.is_file():
        stat = path.stat()
        row.update({'exists': True, 'size_bytes': stat.st_size, 'sha256': sha256_file(path), 'created_at_utc': datetime.fromtimestamp(stat.st_ctime, timezone.utc).isoformat(), 'modified_at_utc': datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat()})
    else:
        row.update({'exists': False, 'size_bytes': np.nan, 'sha256': '', 'created_at_utc': '', 'modified_at_utc': ''})
    artifact_rows.append(row)

artifacts_table = pd.DataFrame(artifact_rows)
ARTIFACTS_PATH = REPORT_ROOT / 'artifacts_v2_checksums.csv'
artifacts_table.to_csv(ARTIFACTS_PATH, index=False)
assert artifacts_table['exists'].all(), 'A frozen V2 artifact is missing.'
display(artifacts_table[['artifact_name', 'size_bytes', 'sha256', 'freeze_policy']])
print('Checksums:', ARTIFACTS_PATH)


,artifact_name,size_bytes,sha256,freeze_policy
0,video_manifest,206824,cebf49f1682859e68850fd8209d6701e22d6690f8bb91c...,reference_only__do_not_overwrite
1,fixed_video_split,71143,64f398f7257b419e7d8dd5c72356214d5cd1f4bce70abd...,reference_only__do_not_overwrite
2,multipos_sequence_manifest,786519,cf12b4964b7c6cfde5f4d9eb2987a19ed292c82260700f...,reference_only__do_not_overwrite
3,multipos_frame_cache_index,8447294,2abbe166dfa5747a271fc1675a6271f0ed8923983a69c9...,reference_only__do_not_overwrite
4,multipos_feature_cache,51214517,78b5a1962d57eb3bb0aff4376129eef7e0d22683ce23b8...,reference_only__do_not_overwrite
5,a2mp_checkpoint,15705,f4ae77cbacb466bcdd6ea4f11951f1f585d4ab6060c30d...,reference_only__do_not_overwrite
6,a2mp_clip_metrics,1473,e1f73f3e0e0c13d6683fcdb084b070a8855e33e284db92...,reference_only__do_not_overwrite
7,a2mp_full_mp4_metrics,823,195fd2e0a521815c367292ac934f23d4a0d2bb45524d68...,reference_only__do_not_overwrite
8,a2mp_full_mp4_predictions,24510,e8adc484714951718a272a0effd47996a9f5098c04413a...,reference_only__do_not_overwrite
9,a2mp_calibration,2588,351f5ecfd044de8b2103db457c62d5a4fa5b8e1bf0ef1b...,reference_only__do_not_overwrite


Checksums: P:\NexarCollisionData\reports_v3\artifacts_v2_checksums.csv


In [3]:
# 2) Registry of V3 experiments
full_reference = json.loads(A2MP_FULL_METRICS_PATH.read_text(encoding='utf-8'))
full_metrics = full_reference['selected_metrics']
REGISTRY_PATH = REPORT_ROOT / 'experiments_v3_registry.csv'
reference_row = {
    'run_id': 'V2_A2MP_REFERENCE', 'stage': 'V2 frozen reference', 'model_id': 'A2-MP',
    'dataset_version': 'v2_multipos', 'split_version': 'metadata_split_v1',
    'window_version': '5s_stride2.5s_top3_mean', 'feature_version': 'resnet18_imagenet_features_v2_multipos_16x224x320',
    'augmentation_version': 'frozen_v2_cache', 'checkpoint_path': str(A2MP_CHECKPOINT_PATH),
    'config_path': 'notebooks/20_v2_multipos_resnet18_meanmax_pooling.ipynb', 'git_commit': current_git_commit(),
    'status': 'frozen_reference', 'primary_metric': 'full_mp4_accident_f1', 'primary_value': full_metrics['f1'],
    'notes': 'Recall={:.6f}; PR-AUC={:.6f}; threshold={:.2f}.'.format(full_metrics['recall'], full_metrics['pr_auc'], full_reference['selected_threshold_by_validation_f1_under_recall_constraint'])
}
audit_row = {
    'run_id': 'V3_00_AUDIT', 'stage': 'V3-0 freeze_and_audit', 'model_id': 'none',
    'dataset_version': 'v2_frozen_reference', 'split_version': 'metadata_split_v1',
    'window_version': 'reference_only', 'feature_version': 'reference_only', 'augmentation_version': 'none',
    'checkpoint_path': '', 'config_path': 'notebooks/30_v3_baseline_audit.ipynb', 'git_commit': current_git_commit(),
    'status': 'running', 'primary_metric': 'audit_gate', 'primary_value': np.nan,
    'notes': 'Checksums, leakage audit, distribution audit and head reproduction.'
}
existing = pd.read_csv(REGISTRY_PATH) if REGISTRY_PATH.exists() else pd.DataFrame()
if len(existing):
    existing = existing.loc[~existing['run_id'].isin(['V2_A2MP_REFERENCE', 'V3_00_AUDIT'])]
registry = pd.concat([existing, pd.DataFrame([reference_row, audit_row])], ignore_index=True)
registry.to_csv(REGISTRY_PATH, index=False)
display(registry.tail(5))
print('Registry:', REGISTRY_PATH)


,run_id,stage,model_id,dataset_version,split_version,window_version,feature_version,augmentation_version,checkpoint_path,config_path,git_commit,status,primary_metric,primary_value,notes
0,V2_A2MP_REFERENCE,V2 frozen reference,A2-MP,v2_multipos,metadata_split_v1,5s_stride2.5s_top3_mean,resnet18_imagenet_features_v2_multipos_16x224x320,frozen_v2_cache,P:\NexarCollisionData\models_v2\resnet18_meanm...,notebooks/20_v2_multipos_resnet18_meanmax_pool...,not_available,frozen_reference,full_mp4_accident_f1,0.737589,Recall=0.866667; PR-AUC=0.722670; threshold=0.40.
1,V3_00_AUDIT,V3-0 freeze_and_audit,none,v2_frozen_reference,metadata_split_v1,reference_only,reference_only,none,,notebooks/30_v3_baseline_audit.ipynb,not_available,running,audit_gate,NaN,"Checksums, leakage audit, distribution audit a..."


Registry: P:\NexarCollisionData\reports_v3\experiments_v3_registry.csv


In [4]:
# 3) Leakage, split and cache audit
video_manifest = pd.read_csv(VIDEO_MANIFEST_PATH)
split_table = pd.read_csv(SPLIT_PATH)
sequence_manifest = pd.read_csv(SEQUENCE_MANIFEST_PATH)
frame_index = pd.read_csv(FRAME_INDEX_PATH)
for table in [video_manifest, split_table, sequence_manifest, frame_index]:
    table['video_id'] = table['video_id'].astype(str)

training_notebook_path = PROJECT_ROOT / 'notebooks' / '20_v2_multipos_resnet18_meanmax_pooling.ipynb'
training_notebook = json.loads(training_notebook_path.read_text(encoding='utf-8'))
training_source = '\n'.join(''.join(cell.get('source', [])) for cell in training_notebook['cells'])
dataset_features_only = 'return self.features[source_index], self.labels[source_index], int(source_index)' in training_source

manifest_splits = video_manifest.set_index('video_id')['split'].reindex(split_table['video_id']).to_numpy()
fixed_splits = split_table['split'].to_numpy()
feature_payload = torch.load(FEATURE_CACHE_PATH, map_location='cpu', weights_only=False)
feature_shape_ok = tuple(feature_payload['features'].shape) == (1560, NUM_FRAMES, FEATURE_DIM)
feature_labels_ok = feature_payload['labels'].tolist() == sequence_manifest['label'].astype(int).tolist()
feature_ids_ok = feature_payload['sequence_ids'] == sequence_manifest['sequence_id'].tolist()
invalid_frames = (~frame_index['frame_valid'].astype(str).str.lower().eq('true')).sum()
near_duplicate_rows = video_manifest['near_duplicate_candidate'].fillna(False).astype(str).str.lower().isin(['true', '1', 'yes']).sum()
exact_duplicate_rows = video_manifest['exact_duplicate_group'].notna().sum()
sequence_split_max = sequence_manifest.groupby('video_id')['split'].nunique().max()
frame_split_max = frame_index.groupby('video_id')['split'].nunique().max()
frame_counts = frame_index.groupby('sequence_id').size()

audit_rows = [
    {'check': 'video_manifest_matches_fixed_split', 'status': 'pass' if np.array_equal(manifest_splits, fixed_splits) else 'fail', 'details': 'video_manifest_v2 and metadata_split_v1 use the same split.'},
    {'check': 'video_id_overlap_between_splits', 'status': 'pass' if sequence_split_max == 1 else 'fail', 'details': 'maximum splits per video_id={}'.format(sequence_split_max)},
    {'check': 'frame_cache_split_isolation', 'status': 'pass' if frame_split_max == 1 else 'fail', 'details': 'maximum splits per cached video_id={}'.format(frame_split_max)},
    {'check': 'sixteen_frames_per_sequence', 'status': 'pass' if frame_counts.eq(NUM_FRAMES).all() else 'fail', 'details': 'minimum/maximum frames={}/{}'.format(frame_counts.min(), frame_counts.max())},
    {'check': 'all_cached_frames_valid', 'status': 'pass' if invalid_frames == 0 else 'fail', 'details': 'invalid frames={}'.format(invalid_frames)},
    {'check': 'feature_cache_shape', 'status': 'pass' if feature_shape_ok else 'fail', 'details': 'feature shape={}'.format(tuple(feature_payload['features'].shape))},
    {'check': 'feature_cache_order_and_labels', 'status': 'pass' if feature_labels_ok and feature_ids_ok else 'fail', 'details': 'labels_match={}; sequence_ids_match={}'.format(feature_labels_ok, feature_ids_ok)},
    {'check': 'forbidden_metadata_excluded_from_a2mp_tensor', 'status': 'pass' if dataset_features_only else 'review', 'details': 'Static Dataset check: model receives cached features, label and index only.'},
    {'check': 'exact_duplicate_files', 'status': 'pass' if exact_duplicate_rows == 0 else 'review', 'details': 'rows in exact duplicate groups={}'.format(exact_duplicate_rows)},
    {'check': 'near_duplicate_candidates', 'status': 'pass' if near_duplicate_rows == 0 else 'review', 'details': 'candidate rows={}'.format(near_duplicate_rows)}
]
leakage_audit = pd.DataFrame(audit_rows)
LEAKAGE_PATH = REPORT_ROOT / 'v3_leakage_audit.csv'
leakage_audit.to_csv(LEAKAGE_PATH, index=False)
assert leakage_audit['status'].eq('pass').all(), 'Leakage audit requires review before V3-1.'
display(leakage_audit)
print('Leakage audit:', LEAKAGE_PATH)


,check,status,details
0,video_manifest_matches_fixed_split,pass,video_manifest_v2 and metadata_split_v1 use th...
1,video_id_overlap_between_splits,pass,maximum splits per video_id=1
2,frame_cache_split_isolation,pass,maximum splits per cached video_id=1
3,sixteen_frames_per_sequence,pass,minimum/maximum frames=16/16
4,all_cached_frames_valid,pass,invalid frames=0
5,feature_cache_shape,pass,"feature shape=(1560, 16, 512)"
6,feature_cache_order_and_labels,pass,labels_match=True; sequence_ids_match=True
7,forbidden_metadata_excluded_from_a2mp_tensor,pass,Static Dataset check: model receives cached fe...
8,exact_duplicate_files,pass,rows in exact duplicate groups=0
9,near_duplicate_candidates,pass,candidate rows=0


Leakage audit: P:\NexarCollisionData\reports_v3\v3_leakage_audit.csv


In [5]:
# 4) Train/validation distribution audit
video_manifest['relative_event_position'] = np.where(video_manifest['label'].eq(1), video_manifest['time_of_event'] / video_manifest['duration'], np.nan)
distribution_rows = []
for field in ['duration', 'fps', 'width', 'height', 'relative_event_position']:
    for split, group in video_manifest.groupby('split', sort=True):
        values = pd.to_numeric(group[field], errors='coerce').dropna()
        distribution_rows.append({'audit_type': 'numeric', 'field': field, 'split': split, 'value': '', 'count': len(values), 'mean': values.mean(), 'median': values.median(), 'std': values.std(), 'min': values.min(), 'max': values.max()})

for field in ['label', 'weather', 'light_conditions', 'scene']:
    counts = video_manifest.groupby(['split', field], dropna=False).size().rename('count').reset_index()
    for _, row in counts.iterrows():
        distribution_rows.append({'audit_type': 'categorical', 'field': field, 'split': row['split'], 'value': str(row[field]), 'count': int(row['count']), 'mean': np.nan, 'median': np.nan, 'std': np.nan, 'min': np.nan, 'max': np.nan})

window_counts = sequence_manifest.groupby(['split', 'video_id']).size().rename('windows_per_video').reset_index()
for split, group in window_counts.groupby('split', sort=True):
    values = group['windows_per_video']
    distribution_rows.append({'audit_type': 'numeric', 'field': 'windows_per_video_v2_multipos', 'split': split, 'value': '', 'count': len(values), 'mean': values.mean(), 'median': values.median(), 'std': values.std(), 'min': values.min(), 'max': values.max()})

distribution_audit = pd.DataFrame(distribution_rows)
DISTRIBUTION_PATH = REPORT_ROOT / 'v3_distribution_audit.csv'
distribution_audit.to_csv(DISTRIBUTION_PATH, index=False)
display(pd.crosstab(video_manifest['split'], video_manifest['label']))
display(distribution_audit.loc[distribution_audit['audit_type'].eq('numeric')])
print('Distribution audit:', DISTRIBUTION_PATH)


label,0,1
split,,
train,240,240
validation,60,60


,audit_type,field,split,value,count,mean,median,std,min,max
0,numeric,duration,train,,480,37.976337,40.100000,6.893637,18.000000,60.000000
1,numeric,duration,validation,,120,37.486569,40.133333,7.149071,15.000000,41.161290
2,numeric,fps,train,,480,30.035417,30.000000,0.597835,23.600000,31.000000
3,numeric,fps,validation,,120,30.097500,30.000000,0.311263,29.200000,31.000000
4,numeric,width,train,,480,1280.000000,1280.000000,0.000000,1280.000000,1280.000000
5,numeric,width,validation,,120,1280.000000,1280.000000,0.000000,1280.000000,1280.000000
6,numeric,height,train,,480,720.000000,720.000000,0.000000,720.000000,720.000000
7,numeric,height,validation,,120,720.000000,720.000000,0.000000,720.000000,720.000000
8,numeric,relative_event_position,train,,240,0.498428,0.494954,0.048356,0.097152,0.946667
9,numeric,relative_event_position,validation,,60,0.493258,0.492457,0.041702,0.343588,0.639012


Distribution audit: P:\NexarCollisionData\reports_v3\v3_distribution_audit.csv


In [6]:
# 5) Reproduce the A2-MP trainable head from the frozen ResNet18 features
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

class SequenceFeatureDataset(Dataset):
    def __init__(self, features, labels, indices):
        self.features = features
        self.labels = labels
        self.indices = torch.as_tensor(indices, dtype=torch.long)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, index):
        source_index = self.indices[index]
        return self.features[source_index], self.labels[source_index], int(source_index)

class MeanMaxHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.classifier = nn.Sequential(nn.LayerNorm(FEATURE_DIM * 2), nn.Dropout(0.35), nn.Linear(FEATURE_DIM * 2, 1))
    def forward(self, sequence_features):
        mean_features = sequence_features.mean(dim=1)
        max_features = sequence_features.max(dim=1).values
        return self.classifier(torch.cat([mean_features, max_features], dim=1)).squeeze(1)

def metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {'threshold': float(threshold), 'accuracy': float(accuracy_score(y_true, predictions)), 'precision': float(precision_score(y_true, predictions, zero_division=0)), 'recall': float(recall_score(y_true, predictions, zero_division=0)), 'f1': float(f1_score(y_true, predictions, zero_division=0)), 'roc_auc': float(roc_auc_score(y_true, probabilities)), 'pr_auc': float(average_precision_score(y_true, probabilities)), 'confusion_matrix': confusion_matrix(y_true, predictions).tolist()}

reproduction = {'status': 'not_run', 'scope': 'trainable head only; full-MP4 reproduction with the new head is pending'}
if RUN_A2MP_HEAD_REPRODUCTION:
    set_seed(SEED)
    # The first V2 run created the frozen ResNet18 cache after setting the seed.
    # Replaying its model initialization and one DataLoader base-seed draw restores
    # the RNG state immediately before the trainable A2-MP head was initialized.
    if REPLAY_V2_INITIAL_FEATURE_EXTRACTION_RNG:
        rng_replay_backbone = models.resnet18(weights=None)
        del rng_replay_backbone
        _ = torch.empty((), dtype=torch.int64).random_().item()
    features = feature_payload['features'].float()
    labels = feature_payload['labels'].long()
    train_indices = np.flatnonzero(sequence_manifest['split'].eq('train').to_numpy())
    validation_indices = np.flatnonzero(sequence_manifest['split'].eq('validation').to_numpy())
    train_loader = DataLoader(SequenceFeatureDataset(features, labels, train_indices), batch_size=HEAD_BATCH_SIZE, shuffle=True, num_workers=0)
    validation_loader = DataLoader(SequenceFeatureDataset(features, labels, validation_indices), batch_size=HEAD_BATCH_SIZE, shuffle=False, num_workers=0)
    model = MeanMaxHead().cpu()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    best_pr_auc, best_epoch, stale_epochs, best_state = -np.inf, 0, 0, None
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        for batch_features, batch_labels, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_features), batch_labels.float())
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch_labels)
        model.eval()
        labels_out, probabilities_out = [], []
        with torch.inference_mode():
            for batch_features, batch_labels, _ in validation_loader:
                labels_out.append(batch_labels.numpy())
                probabilities_out.append(torch.sigmoid(model(batch_features)).numpy())
        y_true = np.concatenate(labels_out)
        probabilities = np.concatenate(probabilities_out)
        epoch_metrics = metrics(y_true, probabilities, 0.5)
        history.append({'epoch': epoch, 'train_loss': loss_sum / len(train_loader.dataset), 'validation_pr_auc': epoch_metrics['pr_auc'], 'validation_f1_at_0_5': epoch_metrics['f1'], 'validation_recall_at_0_5': epoch_metrics['recall']})
        if epoch_metrics['pr_auc'] > best_pr_auc:
            best_pr_auc, best_epoch, stale_epochs = epoch_metrics['pr_auc'], epoch, 0
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.inference_mode():
        probabilities = torch.sigmoid(model(features[validation_indices])).numpy()
    y_true = labels[validation_indices].numpy()
    threshold_table = pd.DataFrame([metrics(y_true, probabilities, float(value)) for value in np.round(np.arange(0.10, 0.901, 0.01), 2)])
    safe = threshold_table.loc[threshold_table['recall'].ge(V2_MINIMUM_RECALL)]
    selected = (safe if len(safe) else threshold_table).sort_values(['f1', 'recall', 'precision'], ascending=False).iloc[0]
    reproduced_metrics = metrics(y_true, probabilities, float(selected['threshold']))
    reference_clip = json.loads(A2MP_CLIP_METRICS_PATH.read_text(encoding='utf-8'))['metrics_at_selected_threshold']
    f1_delta = reproduced_metrics['f1'] - reference_clip['f1']
    pr_auc_delta = reproduced_metrics['pr_auc'] - reference_clip['pr_auc']
    passed = abs(f1_delta) <= 0.01 and abs(pr_auc_delta) <= 0.01
    reproduced_checkpoint = V3_MODEL_ROOT / 'a2mp_v3_reproduction_head_best.pt'
    torch.save({'model_state_dict': model.state_dict(), 'epoch': best_epoch, 'validation_pr_auc': best_pr_auc, 'seed': SEED, 'source_feature_cache_sha256': artifacts_table.loc[artifacts_table['artifact_name'].eq('multipos_feature_cache'), 'sha256'].iloc[0]}, reproduced_checkpoint)
    pd.DataFrame(history).to_csv(REPORT_ROOT / 'a2mp_v3_reproduction_head_history.csv', index=False)
    threshold_table.to_csv(REPORT_ROOT / 'a2mp_v3_reproduction_head_threshold_curve.csv', index=False)
    reproduction = {'status': 'pass' if passed else 'review', 'scope': 'trainable head only; full-MP4 reproduction with the new head is pending', 'best_epoch': int(best_epoch), 'checkpoint_path': str(reproduced_checkpoint), 'selected_threshold': float(selected['threshold']), 'reproduced_metrics': reproduced_metrics, 'reference_clip_metrics': reference_clip, 'f1_delta': float(f1_delta), 'pr_auc_delta': float(pr_auc_delta), 'tolerance': 0.01, 'rng_replay_v2_initial_feature_extraction': bool(REPLAY_V2_INITIAL_FEATURE_EXTRACTION_RNG)}

REPRODUCTION_PATH = REPORT_ROOT / 'v3_baseline_reproduction.json'
REPRODUCTION_PATH.write_text(json.dumps(reproduction, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(reproduction, ensure_ascii=False, indent=2))


{
  "status": "review",
  "scope": "trainable head only; full-MP4 reproduction with the new head is pending",
  "best_epoch": 9,
  "checkpoint_path": "P:\\NexarCollisionData\\models_v3\\a2mp_v3_reproduction_head_best.pt",
  "selected_threshold": 0.36,
  "reproduced_metrics": {
    "threshold": 0.36,
    "accuracy": 0.75,
    "precision": 0.7142857142857143,
    "recall": 0.8333333333333334,
    "f1": 0.7692307692307693,
    "roc_auc": 0.7811111111111111,
    "pr_auc": 0.7425982687763731,
    "confusion_matrix": [
      [
        40,
        20
      ],
      [
        10,
        50
      ]
    ]
  },
  "reference_clip_metrics": {
    "threshold": 0.3,
    "accuracy": 0.7333333333333333,
    "precision": 0.6891891891891891,
    "recall": 0.85,
    "f1": 0.7611940298507462,
    "roc_auc": 0.7808333333333333,
    "pr_auc": 0.7304503474487104,
    "confusion_matrix": [
      [
        37,
        23
      ],
      [
        9,
        51
      ]
    ]
  },
  "f1_delta": 0.0080367393800230

In [7]:
# 6) V3-0 report and gate status
audit_passed = leakage_audit['status'].eq('pass').all() and artifacts_table['exists'].all()
head_passed = reproduction['status'] == 'pass'
full_mp4_status = 'pending_separately_approved_cpu_run' if not RUN_FULL_MP4_REPRODUCTION else 'must_be_run_before_closing_v3_0'
gate_status = 'partial_pass__full_mp4_reproduction_pending' if audit_passed and head_passed else 'blocked_or_review_required'
report_lines = [
    '# V3-0 — Freeze and audit report',
    '',
    'Generated: {}'.format(datetime.now(timezone.utc).isoformat()),
    '',
    '## Frozen reference',
    '- Model: A2-MP frozen ResNet18 + mean-max pooling',
    '- Full-MP4 validation F1: {:.6f}'.format(full_metrics['f1']),
    '- Full-MP4 validation Recall: {:.6f}'.format(full_metrics['recall']),
    '- Full-MP4 validation PR-AUC: {:.6f}'.format(full_metrics['pr_auc']),
    '- Aggregation / threshold: {} / {:.2f}'.format(full_reference['selected_aggregation'], full_reference['selected_threshold_by_validation_f1_under_recall_constraint']),
    '',
    '## Audit result',
    '- Artifact inventory and SHA-256: {}'.format('PASS' if artifacts_table['exists'].all() else 'FAIL'),
    '- Leakage/split/cache audit: {}'.format('PASS' if audit_passed else 'REVIEW'),
    '- Head reproduction: {}'.format(reproduction['status'].upper()),
    '- Head reproduction deltas: F1={:.6f}; PR-AUC={:.6f}'.format(reproduction.get('f1_delta', np.nan), reproduction.get('pr_auc_delta', np.nan)),
    '- Legacy limitation: V2 did not record the exact RNG state immediately before head initialization; a run outside the 0.01 tolerance remains a review item, not a confirmed improvement.',
    '- Full-MP4 reproduction using the new head: {}'.format(full_mp4_status),
    '- V3-0 gate: {}'.format(gate_status),
    '',
    '## Protocol note',
    'V2 clip threshold selection used Recall >= {:.2f}. V3 future model selection uses Accident Recall >= {:.2f}. The frozen full-MP4 reference Recall {:.6f} passes the V3 constraint.'.format(V2_MINIMUM_RECALL, V3_MINIMUM_RECALL, full_metrics['recall']),
    '',
    '## Produced files',
    '- {}'.format(ARTIFACTS_PATH),
    '- {}'.format(REGISTRY_PATH),
    '- {}'.format(LEAKAGE_PATH),
    '- {}'.format(DISTRIBUTION_PATH),
    '- {}'.format(REPRODUCTION_PATH)
]
REPORT_PATH = REPORT_ROOT / 'v3_baseline_reproduction_report.md'
REPORT_PATH.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
registry = pd.read_csv(REGISTRY_PATH)
registry.loc[registry['run_id'].eq('V3_00_AUDIT'), ['status', 'primary_value', 'notes']] = [gate_status, float(reproduction.get('reproduced_metrics', {}).get('f1', np.nan)), 'Audit={}; head reproduction={}; full-MP4 new-head reproduction={}.'.format('pass' if audit_passed else 'review', reproduction['status'], full_mp4_status)]
registry.to_csv(REGISTRY_PATH, index=False)
print('\n'.join(report_lines))
print('Report:', REPORT_PATH)


# V3-0 — Freeze and audit report

Generated: 2026-07-31T19:29:57.758364+00:00

## Frozen reference
- Model: A2-MP frozen ResNet18 + mean-max pooling
- Full-MP4 validation F1: 0.737589
- Full-MP4 validation Recall: 0.866667
- Full-MP4 validation PR-AUC: 0.722670
- Aggregation / threshold: top3_mean / 0.40

## Audit result
- Artifact inventory and SHA-256: PASS
- Leakage/split/cache audit: PASS
- Head reproduction: REVIEW
- Head reproduction deltas: F1=0.008037; PR-AUC=0.012148
- Legacy limitation: V2 did not record the exact RNG state immediately before head initialization; a run outside the 0.01 tolerance remains a review item, not a confirmed improvement.
- Full-MP4 reproduction using the new head: pending_separately_approved_cpu_run
- V3-0 gate: blocked_or_review_required

## Protocol note
V2 clip threshold selection used Recall >= 0.80. V3 future model selection uses Accident Recall >= 0.85. The frozen full-MP4 reference Recall 0.866667 passes the V3 constraint.

## Produced files
-